# Module 04: Supervised Classification / Regression Modeling & Tuning
## Benchmark Multi-Model Comparison, Hyperparameter Tuning & Pipeline Serialization

This notebook implements all required steps for **Module 04**:
1. **Dataset Ingestion & Leak-Free Split:** Preprocessed tabular loan default dataset partitioned with Stratified Train/Test split.
2. **4 Distinct Architectures:**
   - Logistic Regression (Regularized Linear Baseline)
   - Random Forest Classifier (Bagging Ensemble)
   - Gradient Boosting Classifier (Boosting Ensemble)
   - Multi-Layer Perceptron (Neural Network Baseline)
3. **Hyperparameter Optimization:** `GridSearchCV` configured with 5-Fold `StratifiedKFold` cross-validation.
4. **Comprehensive Evaluation:** Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrices.
5. **Champion Selection & Serialization:** Automating champion selection and serializing the model artifact as `.joblib` and `.pkl`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

# Ensure reproducibility
np.random.seed(42)

## 1. Tabular Dataset & Leak-Free Stratified Split

In [ ]:
n_samples = 1500

age = np.random.randint(21, 68, size=n_samples)
annual_income = np.random.exponential(scale=600000, size=n_samples) + 200000
credit_score = np.clip(np.random.normal(loc=680, scale=80, size=n_samples), 350, 850)
debt_to_income_ratio = np.random.uniform(0.05, 0.65, size=n_samples)
open_credit_lines = np.random.poisson(lam=4, size=n_samples) + 1
home_ownership = np.random.choice(['RENT', 'OWN', 'MORTGAGE'], size=n_samples, p=[0.45, 0.20, 0.35])
loan_purpose = np.random.choice(
    ['DEBT_CONSOLIDATION', 'HOME_IMPROVEMENT', 'BUSINESS', 'EDUCATION', 'PERSONAL'],
    size=n_samples, p=[0.40, 0.20, 0.15, 0.15, 0.10]
)
employment_type = np.random.choice(
    ['SALARIED', 'SELF_EMPLOYED', 'FREELANCE', 'UNEMPLOYED'],
    size=n_samples, p=[0.55, 0.25, 0.15, 0.05]
)

latent_risk = (
    - 0.0035 * credit_score
    + 4.8 * debt_to_income_ratio
    - 0.0000015 * annual_income
    + (home_ownership == 'RENT') * 0.45
    + (employment_type == 'UNEMPLOYED') * 1.30
    + (loan_purpose == 'BUSINESS') * 0.40
)
prob_default = 1 / (1 + np.exp(-(latent_risk - np.median(latent_risk))))
default_label = (np.random.uniform(0, 1, size=n_samples) < prob_default).astype(int)

df = pd.DataFrame({
    'age': age,
    'annual_income': annual_income,
    'credit_score': credit_score,
    'debt_to_income_ratio': debt_to_income_ratio,
    'open_credit_lines': open_credit_lines,
    'home_ownership': home_ownership,
    'loan_purpose': loan_purpose,
    'employment_type': employment_type,
    'default_risk': default_label
})

# Add missingness
df.loc[df.sample(frac=0.05, random_state=101).index, 'annual_income'] = np.nan
df.loc[df.sample(frac=0.03, random_state=102).index, 'credit_score'] = np.nan
df.loc[df.sample(frac=0.04, random_state=103).index, 'employment_type'] = np.nan

# Split before any transformation to guarantee zero leakage
X = df.drop(columns=['default_risk'])
y = df['default_risk']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 2. Reusable Preprocessing Transformers

In [ ]:
numeric_features = ['age', 'annual_income', 'credit_score', 'debt_to_income_ratio', 'open_credit_lines']
categorical_features = ['home_ownership', 'loan_purpose', 'employment_type']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 3. Train 4 Distinct Model Architectures with GridSearchCV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_configs = {
    'Logistic Regression': {
        'pipeline': Pipeline([('preprocessor', preprocessor), ('clf', LogisticRegression(random_state=42, max_iter=1000))]),
        'params': {'clf__C': [0.1, 1.0, 10.0], 'clf__penalty': ['l2']}
    },
    'Random Forest': {
        'pipeline': Pipeline([('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))]),
        'params': {'clf__n_estimators': [100, 200], 'clf__max_depth': [5, 10, None], 'clf__min_samples_split': [2, 5]}
    },
    'Gradient Boosting': {
        'pipeline': Pipeline([('preprocessor', preprocessor), ('clf', GradientBoostingClassifier(random_state=42))]),
        'params': {'clf__n_estimators': [100, 150], 'clf__learning_rate': [0.05, 0.1], 'clf__max_depth': [3, 5]}
    },
    'MLP Neural Net': {
        'pipeline': Pipeline([('preprocessor', preprocessor), ('clf', MLPClassifier(random_state=42, max_iter=500))]),
        'params': {'clf__hidden_layer_sizes': [(50,), (50, 25)], 'clf__alpha': [0.001, 0.01]}
    }
}

results = []
fitted_models = {}
roc_curves_data = {}

for name, config in model_configs.items():
    print(f"Optimizing {name} with 5-Fold Stratified CV...")
    grid = GridSearchCV(config['pipeline'], config['params'], cv=cv, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    best_est = grid.best_estimator_
    fitted_models[name] = best_est
    
    # Predictions on holdout test set
    y_pred = best_est.predict(X_test)
    y_prob = best_est.predict_proba(X_test)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_curves_data[name] = (fpr, tpr)
    
    results.append({
        'Model': name,
        'Best Params': str(grid.best_params_),
        'CV ROC-AUC': grid.best_score_,
        'Test ROC-AUC': roc_auc_score(y_test, y_prob),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

## 4. Model Comparison Table & ROC-AUC Curves

In [ ]:
comparison_df = pd.DataFrame(results).sort_values(by='Test ROC-AUC', ascending=False).reset_index(drop=True)
display(comparison_df)

# Plot Multi-Model ROC Curves
plt.figure(figsize=(9, 6))
for name, (fpr, tpr) in roc_curves_data.items():
    auc_val = comparison_df.loc[comparison_df['Model'] == name, 'Test ROC-AUC'].values[0]
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Comparison Curves', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Champion Model Selection, Confusion Matrix & Serialization

In [ ]:
champion_name = comparison_df.iloc[0]['Model']
champion_model = fitted_models[champion_name]
print(f"Selected Champion Model: {champion_name}")

# Confusion Matrix for Champion Model
y_champion_pred = champion_model.predict(X_test)
cm = confusion_matrix(y_test, y_champion_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Default', 'Default'], yticklabels=['Non-Default', 'Default'])
plt.title(f'Confusion Matrix: {champion_name}', fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# Full Classification Report
print("=== Champion Classification Report ===")
print(classification_report(y_test, y_champion_pred, target_names=['Non-Default', 'Default']))

# Serialization to .joblib and .pkl
joblib.dump(champion_model, 'champion_model.joblib')
joblib.dump(champion_model, 'champion_model.pkl')
print("Champion pipeline successfully serialized to 'champion_model.joblib' and 'champion_model.pkl'!")